# 05 — ScaleSC GPU Harmonized: Lung65k

**Run on Libra H100 GPU.**

ScaleSC is a clustering-first pipeline. No native Wilcoxon DE — validation uses standardized DE.

Key differences from Scanpy/rsc:
- ScaleSC does its own QC/filtering internally, but we feed it pre-filtered canonical data
- Neighbors default to `cagra` (approximate), not brute-force
- No harmony step since this is single-sample data
- Uses `X_pca` directly (not `X_pca_harmony`)

In [ ]:
import json, time, os
import numpy as np
import pandas as pd
import scanpy as sc
import scalesc as ssc

print(f"ScaleSC: {getattr(ssc, '__version__', 'UNKNOWN')}")
print(f"Scanpy: {sc.__version__}")
os.makedirs("write", exist_ok=True)

In [ ]:
with open("benchmark_config.json") as f:
    cfg = json.load(f)
gcfg = cfg["global"]
dcfg = cfg["datasets"]["lung65k"]
prefix = dcfg["pipeline_prefix"]
timings = {}

## Prepare ScaleSC input folder

ScaleSC expects a folder containing h5ad file(s). Copy canonical h5ad into a dedicated folder.

In [ ]:
import shutil

scalesc_dir = f"write/scalesc_input_{prefix}"
os.makedirs(scalesc_dir, exist_ok=True)

dest = os.path.join(scalesc_dir, os.path.basename(dcfg["canonical_h5ad"]))
if not os.path.exists(dest):
    shutil.copy2(dcfg["canonical_h5ad"], dest)
    print(f"Copied {dcfg['canonical_h5ad']} -> {dest}")
else:
    print(f"Already exists: {dest}")

## Initialize ScaleSC

In [ ]:
t0 = time.time()
scalesc = ssc.ScaleSC(
    data_dir=scalesc_dir,
    max_cell_batch=1e5,
    preload_on_cpu=True,
    preload_on_gpu=True,
    save_raw_counts=False,
    save_norm_counts=False,
    output_dir="write",
)
timings["init"] = time.time() - t0
print(f"ScaleSC initialized in {timings['init']:.1f}s")
print(f"Shape: {scalesc.adata.shape}")

## HVG

Must run BEFORE normalize_log1p — ScaleSC's seurat_v3 HVG expects raw counts.

In [ ]:
t0 = time.time()
scalesc.highly_variable_genes(n_top_genes=dcfg["n_top_genes"], method="seurat_v3")
timings["hvg"] = time.time() - t0
n_hvg = scalesc.adata.var["highly_variable"].sum()
print(f"HVG ({n_hvg} genes): {timings['hvg']:.1f}s")

## Normalize + log1p

In [ ]:
t0 = time.time()
scalesc.normalize_log1p(target_sum=gcfg["target_sum"])
timings["normalize_log1p"] = time.time() - t0
print(f"Normalize + log1p: {timings['normalize_log1p']:.1f}s")

## PCA

In [ ]:
t0 = time.time()
scalesc.pca(n_components=gcfg["pca_n_comps"], hvg_var="highly_variable")
timings["pca"] = time.time() - t0
print(f"PCA: {timings['pca']:.1f}s")

## Neighbors

ScaleSC defaults to `cagra` (approximate GPU-accelerated graph search).
We use `X_pca` directly since this is single-sample data (no harmony needed).

In [ ]:
t0 = time.time()
scalesc.neighbors(
    n_neighbors=dcfg["n_neighbors"],
    n_pcs=dcfg["neighbors_n_pcs"],
    use_rep="X_pca",
    algorithm="cagra",
)
timings["neighbors"] = time.time() - t0
print(f"Neighbors (cagra): {timings['neighbors']:.1f}s")

## Leiden

In [ ]:
t0 = time.time()
scalesc.leiden(
    resolution=dcfg["leiden_resolution"],
    random_state=gcfg["random_state"],
)
timings["leiden"] = time.time() - t0
n_clusters = scalesc.adata.obs["leiden"].nunique()
print(f"Leiden ({n_clusters} clusters): {timings['leiden']:.1f}s")

## UMAP

In [ ]:
t0 = time.time()
scalesc.umap(random_state=gcfg["random_state"])
timings["umap"] = time.time() - t0
print(f"UMAP: {timings['umap']:.1f}s")

## Save outputs

ScaleSC has no native Wilcoxon DE. Validation uses standardized DE on canonical matrix.

In [ ]:
out = f"write/{prefix}_scalesc_gpu_harmonized"
adata = scalesc.adata

# Clusters
pd.DataFrame({
    "barcode": adata.obs_names.astype(str),
    "leiden": adata.obs["leiden"].astype(str).values,
}).to_csv(f"{out}_clusters.csv", index=False)

# UMAP
if "X_umap" in adata.obsm:
    pd.DataFrame(
        adata.obsm["X_umap"],
        index=adata.obs_names.astype(str),
        columns=["UMAP_1", "UMAP_2"],
    ).reset_index(names="barcode").to_csv(f"{out}_umap.csv", index=False)

# Save h5ad (without count matrix — ScaleSC stores it separately)
scalesc.save(data_name=f"{prefix}_scalesc_gpu_harmonized")

# Spec
spec = {
    "pipeline": "scalesc_gpu_harmonized",
    "dataset": dcfg["name"],
    "scalesc_version": getattr(ssc, "__version__", "UNKNOWN"),
    "neighbors_algorithm": "cagra",
    "native_wilcoxon_de": False,
    "timings_seconds": timings,
    "parameters": {
        "target_sum": gcfg["target_sum"],
        "n_top_genes": dcfg["n_top_genes"],
        "pca_n_comps": gcfg["pca_n_comps"],
        "n_neighbors": dcfg["n_neighbors"],
        "neighbors_n_pcs": dcfg["neighbors_n_pcs"],
        "neighbors_algorithm": "cagra",
        "leiden_resolution": dcfg["leiden_resolution"],
        "random_state": gcfg["random_state"],
    },
    "results": {
        "n_cells": int(adata.n_obs),
        "n_genes": int(adata.n_vars),
        "n_hvg": int(n_hvg),
        "n_clusters": n_clusters,
    },
}
with open(f"{out}_spec.json", "w") as f:
    json.dump(spec, f, indent=2)

print("\n=== Timings ===")
total = 0
for step, t in timings.items():
    print(f"  {step:20s}: {t:8.1f}s")
    total += t
print(f"  {'TOTAL':20s}: {total:8.1f}s ({total/60:.1f} min)")
print(f"\nClusters: {n_clusters}")
print(f"\nNote: No native DE. Run validation notebook for standardized DE comparison.")